In [4]:
import os
import sys
import pickle

import pandas as pd
import sqlite3
import cv2
import numpy as np

import group_images

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
DB_PATH = "../people_detection.db"
SEQ_CSV_PATH = "../seq_files.csv"

In [6]:
if os.path.exists(SEQ_CSV_PATH):
    df = pd.read_csv(SEQ_CSV_PATH)
else:
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(
        """SELECT DISTINCT seq_name, seq_path, avi_path, frame_h, frame_w, camera_type, altitude
        FROM boxes;
        """, conn)
    df.to_csv("../seq_files.csv", index=False)

print(df.shape)
df.head()

(9634, 7)


,seq_name,seq_path,avi_path,frame_h,frame_w,camera_type,altitude
0,Noise_1_enter_exit_NoSw_28_16_15_11_2015.avi.seq,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,720.0,1280.0,NaN,253
1,PG1_dev_H660_Walk_Q_BlackCarpet_NY_Q_150928576...,/mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev2_h...,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,960.0,1280.0,SunPlus_PG_Camera,660
2,PG1_DL_H300_Walking_NY_St1707_1500538480.avi.seq,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,960.0,1280.0,SunPlus_PG1_Camera1,300
3,PG1_dev_H270_VLC_3mHall_Door_05Wall_1509468343...,/mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev_VL...,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,960.0,1280.0,SunPlus_PG_Camera,270
4,PG1_H300_MR_SquareSeat_NY_15stnd_1509274685.av...,/mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev_DL...,/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling...,960.0,1280.0,SunPlus_PG_Camera,300


In [7]:
avi_pth = df.iloc[0]["avi_path"]
avi_pth

'/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling_POV/Occupancy/OCS_people_counting_H253_B_PIRar/Noise_1_enter_exit_NoSw_28_16_15_11_2015.avi'

In [8]:
RES_FOLDER_PATH = "embeddings"
os.makedirs(RES_FOLDER_PATH, exist_ok=True)

In [15]:
def process_avi(avi_pth):
    try:
        N = 30

        cap = cv2.VideoCapture(avi_pth)

        frames = []
        count = -1
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            count += 1
            if not count % N:
                frames.append(frame)

        cap.release()

        embedder = group_images.Embedder()
        embeddings = [embedder.extract_embedding_from_img(frame).detach().numpy() for frame in frames]

        file_path = os.path.join(RES_FOLDER_PATH, os.path.splitext(os.path.basename(avi_pth))[0])
        with open(file_path, "wb") as f:
            np.save(f, embeddings)

        return avi_pth, "OK"
    except Exception as e:
        return avi_pth, repr(e)

In [16]:
process_avi("/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling_POV/PG1_DL/PG1_DL_training_POS08_Desk/PG1_DL_H300_Office_Desk_NY_17_12_37_23_2017.avi")

Device: cuda


('/mnt/pg-fs01/Current/DB/Home_Appliance/Ceiling_POV/PG1_DL/PG1_DL_training_POS08_Desk/PG1_DL_H300_Office_Desk_NY_17_12_37_23_2017.avi',
 'OK')

In [17]:
process_avi("/mnt/pg-fs02/Public/DB_Team/PG1/0_Orange_IT/0_Paid_UNTIL_2017_06_20_1500$_AND_3X850$_move2db/2017_05_17_H300_DL_NY_Tag_145057_WithPos_move2pos06/done_db_pose/PG1_DL_H300_Office_desk_NY_17_12_37_23_2017.avi")

Device: cuda


('/mnt/pg-fs02/Public/DB_Team/PG1/0_Orange_IT/0_Paid_UNTIL_2017_06_20_1500$_AND_3X850$_move2db/2017_05_17_H300_DL_NY_Tag_145057_WithPos_move2pos06/done_db_pose/PG1_DL_H300_Office_desk_NY_17_12_37_23_2017.avi',
 'OK')

In [18]:
process_avi("/mnt/pg-fs02/Public/DB_Team/PG1/Benchmarks/0_To_Mark/PG1_2018_03_29_H655_Office_HD_NY/done/PG1_H655_dev_Office_HD_NY_180320-180758.avi")

Device: cuda


('/mnt/pg-fs02/Public/DB_Team/PG1/Benchmarks/0_To_Mark/PG1_2018_03_29_H655_Office_HD_NY/done/PG1_H655_dev_Office_HD_NY_180320-180758.avi',
 'OK')